# Bayesian Knowledge Tracing (BKT) 

This notebook applies **Bayesian Knowledge Tracing** using the [pyBKT](https://github.com/CAHLR/pyBKT) library to model student mastery over time.

Two models are compared:
- **Fine KC model** — trained on fine-grained knowledge components (original skill tags)
- **Modeling KC model** — trained on higher-level aggregated knowledge components

Each model is evaluated on a held-out validation set using **RMSE** and **AUC**.

## Imports and Data


In [1]:
import pandas as pd
from pathlib import Path
from pyBKT.models import Model
import sys
sys.path.append('..')
from src.preprocess_split import split 

In [2]:
base_path = Path().resolve().parent
raw_path = base_path / 'data' / 'raw'
output_path = base_path / 'data' / 'output'

In [3]:
# Student observations - One row per student–question attempt, including correctness and KC tag
obs = pd.read_excel(
    raw_path / 'Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx', 
    sheet_name='Student_Observations')


# Fine KC to Modeling KC mapping - Maps each fine-grained KC to a higher-level modeling KC
kc_map = pd.read_excel(
    raw_path / "mkc_mapping_pack_v1.0..xlsx",
    sheet_name="FineKC_to_ModelingKC_Map"
)


# Modeling KC weights - Importance weights assigned to each modeling KC
weights = pd.read_excel(
    raw_path / "mkc_weights_dataset.xlsx",
    sheet_name="MKC_Weights"
)

## Data Preparation

### Slim down the KC mapping table

Only the columns needed for joining and reporting are kept from `kc_map`, and duplicate rows are dropped.

In [4]:
kc_map_slim = kc_map[[
    "fine_kc_id",
    "fine_kc_label",
    "modeling_kc_id",
    "modeling_kc_label",
    "modeling_unit",
    "fine_reporting_group"
]].drop_duplicates()

### Join observations with KC mapping

Each student observation is enriched with its corresponding modeling KC label and unit by joining on `primary_kc_id` → `fine_kc_id`. 

In [5]:
df = obs.merge(
    kc_map_slim,
    left_on="primary_kc_id",
    right_on="fine_kc_id",
    how="left"         
)

### Join with modeling KC weights

Importance weights are attached to each row via `modeling_kc_id`.

In [6]:
df_2 = df.merge(
    weights,
    on="modeling_kc_id",
    how="left"
)

## Fine KC Model

The dataset is split into train / test / validation sets (using the default fine KC skill column), then a BKT model is fitted and used to generate predictions on the test set.


In [7]:
kc_train, kc_test, kc_validation = split(df_2)
kc_model = Model(seed = 42, num_fits = 1)
kc_model.fit(data = kc_train)
kc_preds = kc_model.predict(data=kc_test)

# Uncomment the following line to save predictions
# kc_preds.to_csv(output_path / 'kc_preds.csv')

## Modeling KC Model

The same procedure is repeated, but the skill column is set to `modeling_kc_id` — the higher-level aggregated KC. This reduces the number of skills the model needs to learn parameters for, which can improve generalisation when individual fine KCs have sparse data.

In [8]:
mkc_train, mkc_test, mkc_validation = split(df_2, 'modeling_kc_id')
mkc_model = Model(seed = 42, num_fits = 1)
mkc_model.fit(data = mkc_train)
mkc_preds = mkc_model.predict(data=mkc_test)

# Uncomment the following line to save predictions
# mkc_preds.to_csv(output_path / 'mkc_preds.csv')

## Evaluation

Both models are evaluated on the held-out validation set using two metrics:

- **RMSE** (Root Mean Squared Error) — measures prediction accuracy; lower is better
- **AUC** (Area Under the ROC Curve) — measures discrimination ability; higher is better (0.5 = random, 1.0 = perfect)

### Fine KC model

In [9]:
kc_rmse = kc_model.evaluate(data = kc_validation)
kc_auc = kc_model.evaluate(data = kc_validation, metric = 'auc')
print("Training RMSE: %f" % kc_rmse)
print("Training AUC: %f" % kc_auc)

Training RMSE: 0.497632
Training AUC: 0.575974


### Modeling KC model

In [10]:
mkc_rmse = mkc_model.evaluate(data = mkc_validation)
mkc_auc = mkc_model.evaluate(data = mkc_validation, metric = 'auc')
print("Training RMSE: %f" % mkc_rmse)
print("Training AUC: %f" % mkc_auc)

Training RMSE: 0.495454
Training AUC: 0.580397


## Results Summary

| Model | RMSE | AUC |
|---|---|---|
| Fine KC | 0.4977 | 0.5760 |
| Modeling KC | 0.4955 | 0.5803 |

The **Modeling KC model** edges out the Fine KC model on both metrics, suggesting that aggregating to higher-level skills provides a modest but consistent improvement — likely because pooling observations across related fine KCs gives the model more signal per skill parameter to learn from.